---
title: "IPCC AR6 Top 10 Global South Authors"
format:
  html:
    code-fold: true
execute:
  enabled: false
---

# IPCC AR6 Top 10 Global South Authors

Interactive map showing the top 10 IPCC AR6 authors who are **resident in the Global South**, ranked by number of chapters contributed to. Each author's chapter contributions link directly to their Wikibase items.

**Instructions:**
1. Update the `WIKI_BASE_URL` and `SPARQL_ENDPOINT` variables for your environment
2. Run all cells to generate the map and table
3. Quarto will render the stored outputs without re-executing

**Data Source:** Wikibase SPARQL endpoint + `authors-ar6.csv`  
**Visualisation:** Plotly geographic map + chapter sitelinks table

In [1]:
#| include: false
# Import required libraries
import sys
import socket
import requests
import pandas as pd
import plotly.express as px
from SPARQLWrapper import SPARQLWrapper, JSON
from IPython.display import display, HTML

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Configuration

Update `WIKI_BASE_URL` and `SPARQL_ENDPOINT` to match your environment:

| Environment | Wiki Base URL | SPARQL Endpoint |
|-------------|---------------|-----------------|
| **LOCAL** | `http://localhost:8080` | `http://localhost:9999/bigdata/namespace/wdq/sparql` |
| **DEV** | `https://dev-climatekg.semanticclimate.org` | `https://dev-climatekg.semanticclimate.org/query/proxy/sparql` |
| **TEST** | `https://test-climatekg.semanticclimate.org` | `https://test-climatekg.semanticclimate.org/query/proxy/sparql` |
| **PROD** | `https://prod-climatekg.semanticclimate.org` | `https://prod-climatekg.semanticclimate.org/query/proxy/sparql` |

In [2]:
#| include: false
# ============================================================
# CONFIGURATION - Update these two variables for your environment
# ============================================================

# Wiki base URL (no trailing slash)
WIKI_BASE_URL = "http://localhost:8080"

# SPARQL endpoint URL
SPARQL_ENDPOINT = "http://localhost:9999/bigdata/namespace/wdq/sparql"

# For other environments:
# DEV:  WIKI_BASE_URL = "https://dev-climatekg.semanticclimate.org"
#       SPARQL_ENDPOINT = "https://dev-climatekg.semanticclimate.org/query/proxy/sparql"
# TEST: WIKI_BASE_URL = "https://test-climatekg.semanticclimate.org"
#       SPARQL_ENDPOINT = "https://test-climatekg.semanticclimate.org/query/proxy/sparql"
# PROD: WIKI_BASE_URL = "https://prod-climatekg.semanticclimate.org"
#       SPARQL_ENDPOINT = "https://prod-climatekg.semanticclimate.org/query/proxy/sparql"

# CSV path (relative to this notebook)
CSV_PATH = "../data-xml-dtd/authors-ar6.csv"

print(f"Wiki Base URL:   {WIKI_BASE_URL}")
print(f"SPARQL Endpoint: {SPARQL_ENDPOINT}")

Wiki Base URL:   http://localhost:8080
SPARQL Endpoint: http://localhost:9999/bigdata/namespace/wdq/sparql


## Check SPARQL Endpoint Availability

**Troubleshooting:**
- `docker compose ps` — confirm all containers are running
- LOCAL SPARQL: `http://localhost:9999/bigdata/namespace/wdq/sparql`

In [3]:
#| include: false
# Quick port check
ports_to_check = {
    8080: "MediaWiki/Wikibase (Wiki)",
    8081: "Query UI",
    9999: "WDQS/Blazegraph SPARQL",
}
print("Checking localhost ports...\n")
for port, service in ports_to_check.items():
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(1)
    result = sock.connect_ex(("localhost", port))
    sock.close()
    status = "✓ OPEN " if result == 0 else "✗ CLOSED"
    print(f"{status}  Port {port} - {service}")

# Connectivity test
print("\nTesting SPARQL endpoint...")
try:
    r = requests.post(
        SPARQL_ENDPOINT,
        data={"query": "SELECT * WHERE { ?s ?p ?o } LIMIT 1"},
        headers={"Accept": "application/sparql-results+json"},
        timeout=15
    )
    if r.status_code == 200:
        print("✓ SPARQL endpoint is responding")
    else:
        print(f"✗ Unexpected status: {r.status_code}")
        sys.exit(1)
except Exception as e:
    print(f"✗ Cannot reach SPARQL endpoint: {e}")
    print("Start the stack: docker compose up -d")
    sys.exit(1)

Checking localhost ports...

✓ OPEN   Port 8080 - MediaWiki/Wikibase (Wiki)
✓ OPEN   Port 8081 - Query UI
✓ OPEN   Port 9999 - WDQS/Blazegraph SPARQL

Testing SPARQL endpoint...
✓ SPARQL endpoint is responding


## Query and Process Data

In [4]:
#| echo: false
# ── Step 1: Query Wikibase for author UIDs ──────────────────────────────────
author_query = f"""
PREFIX wdt: <{WIKI_BASE_URL}/prop/direct/>
PREFIX wd:  <{WIKI_BASE_URL}/entity/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?author ?uid ?authorLabel
WHERE {{
    ?author wdt:P1 wd:Q3998 .
    ?author wdt:P20 ?uid .
    OPTIONAL {{ ?author rdfs:label ?authorLabel . FILTER(LANG(?authorLabel) = "en") }}
}}
"""

sparql = SPARQLWrapper(SPARQL_ENDPOINT)
sparql.setQuery(author_query)
sparql.setReturnFormat(JSON)

try:
    results = sparql.query().convert()
    wiki_map = {
        b["uid"]["value"]: b["author"]["value"]
        for b in results["results"]["bindings"]
    }
    print(f"Wikibase author entities: {len(wiki_map):,}")
except Exception as e:
    print(f"Warning: Could not query Wikibase: {e}")
    wiki_map = {}

# ── Step 2: Load CSV ─────────────────────────────────────────────────────────
col_names = [
    "uid", "last_name", "first_name", "gender", "citizenship",
    "country_residence", "affiliation", "report", "chapter",
    "chapter_qid", "role", "source_url", "date_accessed"
]
df = pd.read_csv(CSV_PATH, header=0, names=col_names, skipinitialspace=True)
print(f"CSV rows loaded:          {len(df):,}")

# Build QID -> chapter name lookup
qid_to_name = dict(zip(df["chapter_qid"], df["chapter"]))

# ── Step 3: Define Global North (everything else = Global South) ─────────────
GLOBAL_NORTH = {
    "USA", "Canada", "UK", "United Kingdom", "Germany", "France", "Italy",
    "Spain", "Netherlands", "Belgium", "Switzerland", "Austria", "Sweden",
    "Norway", "Denmark", "Finland", "Iceland", "Ireland", "Portugal", "Greece",
    "Poland", "Czech Republic", "Hungary", "Romania", "Bulgaria", "Croatia",
    "Slovakia", "Slovenia", "Estonia", "Latvia", "Lithuania", "Luxembourg",
    "Malta", "Cyprus", "Monaco", "Australia", "New Zealand", "Japan",
    "Israel", "Singapore", "South Korea", "Republic of Korea", "Taiwan"
}

# Plotly country name normalisation
COUNTRY_NORM = {
    "USA": "United States",
    "UK": "United Kingdom",
    "Czech Republic": "Czechia",
    "Republic of Korea": "South Korea",
    "Cote d'Ivoire": "Ivory Coast",
    "Micronesia, Federated States of": "Micronesia",
    "Türkiye": "Turkey",
    "Tanzania": "Tanzania",
    "Democratic Republic of the Congo": "Democratic Republic of Congo",
}

# ── Step 4: Filter to Global South residents ─────────────────────────────────
df_gs = df[
    ~df["country_residence"].isin(GLOBAL_NORTH) &
    df["country_residence"].notna() &
    (df["country_residence"].str.strip() != "")
].copy()

# ── Step 5: Aggregate chapters per unique author ──────────────────────────────
author_stats = df_gs.groupby("uid").agg(
    last_name=("last_name", "first"),
    first_name=("first_name", "first"),
    country=("country_residence", "first"),
    chapter_count=("chapter_qid", "nunique"),
    chapters=("chapter_qid", lambda x: sorted(x.unique().tolist())),
).reset_index()

author_stats["author_name"] = (
    author_stats["first_name"].str.strip() + " " + author_stats["last_name"].str.strip()
)
author_stats["country_plotly"] = author_stats["country"].map(COUNTRY_NORM).fillna(author_stats["country"])

# Derive chapter names aligned to QID list
author_stats["chapter_names"] = author_stats["chapters"].apply(
    lambda qids: [qid_to_name.get(q, q) for q in qids]
)

# ── Step 6: Top 10 by chapter count ──────────────────────────────────────────
top10 = author_stats.nlargest(10, "chapter_count").reset_index(drop=True)
top10["rank"] = range(1, 11)

print(f"Global South authors (resident): {len(author_stats):,}")
print(f"\nTop 10 by chapter contributions:")
for _, row in top10.iterrows():
    print(f"  {row['rank']:2}. {row['author_name']:<30} {row['country']:<25} {row['chapter_count']} chapters")

Wikibase author entities: 932
CSV rows loaded:          1,164
Global South authors (resident): 351

Top 10 by chapter contributions:
   1. Amjad Abdulla                  Maldives                  4 chapters
   2. Fatima Denton                  Ghana                     4 chapters
   3. Aditi Mukherji                 Kenya                     4 chapters
   4. Aromar Revi                    India                     4 chapters
   5. Joyashree Roy                  India                     4 chapters
   6. Paulina Aldunce                Chile                     3 chapters
   7. Gabriel Blanco                 Argentina                 3 chapters
   8. Dipak Dasgupta                 India                     3 chapters
   9. Aïda Diongue-Niang             Senegal                   3 chapters
  10. Rodel Lasco                    Philippines               3 chapters


## Map and Chapter Sitelinks

In [5]:
#| echo: false
# ── Build chapter sitelinks HTML ─────────────────────────────────────────────
def build_chapter_links(chapters, chapter_names, wiki_base):
    links = []
    for qid, name in zip(chapters, chapter_names):
        url = f"{wiki_base}/wiki/Item:{qid}"
        short = (name[:70] + "\u2026") if len(name) > 70 else name
        links.append(
            f'<a href="{url}" target="_blank" '
            f'style="display:block; margin:2px 0; color:#2E86AB">{short}</a>'
        )
    return "".join(links)

top10["chapter_links"] = top10.apply(
    lambda r: build_chapter_links(r["chapters"], r["chapter_names"], WIKI_BASE_URL),
    axis=1
)

# ── Scatter geo map ───────────────────────────────────────────────────────────
fig = px.scatter_geo(
    top10,
    locations="country_plotly",
    locationmode="country names",
    size="chapter_count",
    hover_name="author_name",
    hover_data={
        "country": True,
        "chapter_count": True,
        "country_plotly": False,
        "rank": True
    },
    color="chapter_count",
    color_continuous_scale="Viridis",
    title="Top 10 IPCC AR6 Authors Resident in the Global South",
    size_max=35,
    projection="natural earth"
)
fig.update_layout(
    height=520,
    coloraxis_colorbar_title="Chapters",
    geo=dict(
        showframe=False,
        showcoastlines=True,
        coastlinecolor="#aaaaaa",
        showland=True,
        landcolor="#f5f5f0",
        showocean=True,
        oceancolor="#e8f0f8",
    ),
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

# ── Chapter sitelinks table ───────────────────────────────────────────────────
rows_html = ""
for _, row in top10.iterrows():
    bg = "#f9f9f9" if row["rank"] % 2 == 0 else "#ffffff"
    rows_html += (
        f'<tr style="background:{bg}; vertical-align:top">'
        f'<td style="padding:8px; text-align:center; font-weight:bold">{row["rank"]}</td>'
        f'<td style="padding:8px">{row["author_name"]}</td>'
        f'<td style="padding:8px">{row["country"]}</td>'
        f'<td style="padding:8px; text-align:center">{row["chapter_count"]}</td>'
        f'<td style="padding:8px; font-size:12px">{row["chapter_links"]}</td>'
        f'</tr>'
    )

table_html = f"""
<h3 style="margin-top:1.5em">Chapter Contributions with Wikibase Sitelinks</h3>
<table style="width:100%; border-collapse:collapse; font-size:14px">
  <thead>
    <tr style="background:#2E86AB; color:white">
      <th style="padding:8px; text-align:center">#</th>
      <th style="padding:8px; text-align:left">Author</th>
      <th style="padding:8px; text-align:left">Country of Residence</th>
      <th style="padding:8px; text-align:center">Chapters</th>
      <th style="padding:8px; text-align:left">Chapter Sitelinks (Wikibase)</th>
    </tr>
  </thead>
  <tbody>{rows_html}</tbody>
</table>
"""
display(HTML(table_html))

C:\Users\worthingtons\AppData\Local\Temp\ipykernel_15452\451492935.py:20: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.scatter_geo(


#,Author,Country of Residence,Chapters,Chapter Sitelinks (Wikibase)
1,Amjad Abdulla,Maldives,4,"Chapter 15: Investment and financeChapter 4: Strengthening and implementing the global response to the t…Chapter 6: Interlinkages between desertification, land degradation, fo…Chapter 6: Extremes, Abrupt Changes and Managing Risks"
2,Fatima Denton,Ghana,4,Chapter 17: Accelerating the transition in the context of sustainable …SPMLRChapter 1: Framing and Context
3,Aditi Mukherji,Kenya,4,Chapter 4: WaterSPMLRChapter 2: High Mountain Areas
4,Aromar Revi,India,4,Chapter 18: Climate resilient development pathwaysSPMLRChapter 4: Strengthening and implementing the global response to the t…
5,Joyashree Roy,India,4,"Chapter 5: Demand, services and social aspects of mitigationSPMLRChapter 5: Sustainable development, poverty eradication, and reducing …"
6,Paulina Aldunce,Chile,3,"Chapter 7: Health, wellbeing and the changing structure of communitiesSPMLR"
7,Gabriel Blanco,Argentina,3,"Chapter 16: Innovation, technology development and transferSPMLR"
8,Dipak Dasgupta,India,3,Chapter 15: Investment and financeSPMLR
9,Aïda Diongue-Niang,Senegal,3,"SPMLRChapter 1: Framing, context, methods"
10,Rodel Lasco,Philippines,3,"Chapter 5: Food, fibre, and other ecosystem productsSPMLR"
